In [1]:
import numpy as np
import pandas as pd
from numpy.linalg import norm

lib = pd.read_csv("LIB.csv")
corpus = pd.read_csv("CORPUS.csv")
vocab = pd.read_csv("VOCAB.csv")

In [2]:
bow = (
    corpus[corpus["term_str"].str.match(r"^[a-z]+$", na=False)]
    .groupby(["doc_id", "term_str"])
    .size()
    .reset_index(name="n")
)

In [3]:
bow = bow.merge(
    lib[["doc_id", "author", "title"]],
    on="doc_id",
    how="left"
)

In [4]:
bow = bow.merge(
    vocab[["term_str", "stop", "porter_stem", "max_pos", "max_pos_group"]],
    on="term_str",
    how="left"
)

In [5]:
bow = bow[
    ["doc_id", "author", "title", "term_str", "n", "stop", "porter_stem", "max_pos", "max_pos_group"]
]

In [6]:
bow.to_csv("BOW.csv", index=False)
bow.head()

,doc_id,author,title,term_str,n,stop,porter_stem,max_pos,max_pos_group
0,DOC001,Edgar Allan Poe,The Purloined Letter,a,182,True,a,DT,OTHER
1,DOC001,Edgar Allan Poe,The Purloined Letter,abandon,1,False,abandon,VB,VERB
2,DOC001,Edgar Allan Poe,The Purloined Letter,abernethy,4,False,abernethi,NNP,NOUN
3,DOC001,Edgar Allan Poe,The Purloined Letter,able,1,False,abl,JJ,ADJ
4,DOC001,Edgar Allan Poe,The Purloined Letter,abounds,1,False,abound,VBZ,VERB


In [7]:
bow.shape

(100068, 9)

In [8]:
bow.groupby("author")["n"].sum()

author
Edgar Allan Poe        190086
Nathaniel Hawthorne    292804
Name: n, dtype: int64

In [9]:
bow.columns

Index(['doc_id', 'author', 'title', 'term_str', 'n', 'stop', 'porter_stem',
       'max_pos', 'max_pos_group'],
      dtype='object')

## DTM

In [10]:
dtm = bow.pivot_table(
    index="doc_id",
    columns="term_str",
    values="n",
    fill_value=0
)

In [11]:
dtm_sparse = dtm.astype(pd.SparseDtype("int", 0))

In [12]:
dtm_sparse.to_csv("DTM.csv")

In [13]:
dtm.shape

(65, 18442)

## TFIDF

In [14]:
tf = dtm.div(dtm.sum(axis=1), axis=0)

In [15]:
import numpy as np

N = dtm.shape[0]

df = (dtm > 0).sum(axis=0)

idf = np.log2(N / df)

In [16]:
tfidf = tf * idf

In [17]:
tfidf.to_csv("TFIDF.csv")
tfidf.head()

term_str,a,aaraaf,aback,abandon,abandoned,abandoning,abandonment,abased,abasement,abashed,...,zest,zigzag,zimmerman,zodiac,zodiacal,zohar,zone,zones,zufalle,zusammen
doc_id,,,,,,,,,,,,,,,,,,,,,
DOC001,0.000588,0.0,0.000000,0.000581,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
DOC002,0.000517,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.008183,0.0,0.0,0.0,0.0
DOC003,0.000585,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
DOC004,0.000518,0.0,0.000867,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0
DOC005,0.000480,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0


In [18]:
tfidf.shape

(65, 18442)

## Reduced and Normalized TFIDF_L2

In [19]:
top_terms = (
    vocab[~vocab["stop"]]
    .sort_values("dfidf", ascending=False)
    .head(2000)["term_str"]
)

In [20]:
tfidf_reduced = tfidf[top_terms]

In [21]:
len(top_terms)

2000

In [22]:
from numpy.linalg import norm

tfidf_l2 = tfidf_reduced.div(
    norm(tfidf_reduced, axis=1),
    axis=0
)

In [23]:
tfidf_l2.to_csv("TFIDF_L2.csv")
tfidf_l2.head()

term_str,laid,wonder,gaze,fresh,touch,followed,filled,evidently,ere,ear,...,nearly,small,days,certain,right,feet,whether,threw,truth,suddenly
doc_id,,,,,,,,,,,,,,,,,,,,,
DOC001,0.0,0.000000,0.000000,0.000000,0.000000,0.010566,0.010566,0.000000,0.0,0.000000,...,0.019548,0.004887,0.000000,0.058645,0.019548,0.000000,0.004887,0.004887,0.014661,0.004887
DOC002,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,...,0.000000,0.000000,0.000000,0.027014,0.000000,0.000000,0.000000,0.000000,0.027014,0.000000
DOC003,0.0,0.000000,0.000000,0.000000,0.031096,0.000000,0.015548,0.015548,0.0,0.000000,...,0.028764,0.014382,0.021573,0.014382,0.007191,0.014382,0.000000,0.000000,0.007191,0.007191
DOC004,0.0,0.000000,0.013348,0.013348,0.000000,0.000000,0.000000,0.000000,0.0,0.026696,...,0.037041,0.018520,0.006173,0.000000,0.018520,0.012347,0.012347,0.018520,0.018520,0.030867
DOC005,0.0,0.021989,0.000000,0.000000,0.000000,0.000000,0.021989,0.021989,0.0,0.000000,...,0.040681,0.030511,0.010170,0.010170,0.010170,0.030511,0.020341,0.010170,0.030511,0.010170
